# Analyzing agent telemetry with the BigQuery Agent Analytics SDK

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/bigquery-samples/blob/main/end-to-end/bqaa-sdk/bqaa-sdk-intro.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fbigquery-samples%2Fmain%2Fend-to-end%2Fbqaa-sdk%2Fbqaa-sdk-intro.ipynb?utm_campaign=CDR_0x619bfc4f_user-journey_b578958254&utm_medium=external&utm_source=other">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https%3A%2F%2Fgithub.com%2FGoogleCloudPlatform%2Fbigquery-samples%2Fblob%2Fmain%2Fend-to-end%2Fbqaa-sdk%2Fbqaa-sdk-intro.ipynb&utm_campaign=CDR_0x619bfc4f_user-journey_b578958254&utm_medium=external&utm_source=other">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/bigquery-samples/blob/main/end-to-end/bqaa-sdk/bqaa-sdk-intro.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/bigquery-samples/blob/main/end-to-end/bqaa-sdk/bqaa-sdk-intro.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/bigquery-samples/blob/main/end-to-end/bqaa-sdk/bqaa-sdk-intro.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/bigquery-samples/blob/main/end-to-end/bqaa-sdk/bqaa-sdk-intro.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/bigquery-samples/blob/main/end-to-end/bqaa-sdk/bqaa-sdk-intro.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/bigquery-samples/blob/main/end-to-end/bqaa-sdk/bqaa-sdk-intro.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>

Understanding how your AI agents are performing in production is essential. In this notebook, you'll learn how to analyze your agent's telemetry data with the [BigQuery Agent Analytics (BQAA) SDK](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK). This SDK is intended as a consumption layer for agent data captured using BigQuery Agent Analytics, either via the ADK plugin or LangGraph.

Already having BQAA logs available for your agent is **not** a prerequisite for running this notebook. You'll import a sample dataset that includes 30 days of telemetry data from a retail agent. If you want to dive into the agent code for the agents used to generate these metrics, see [this codelab](https://codelabs.developers.google.com/adk-bigquery-agent-analytics-plugin?hl=en&utm_campaign=CDR_0x619bfc4f_user-journey_b578958254&utm_medium=external&utm_source=other#4).

If you prefer, you can also run this notebook on your own BQAA datasets. You just need to replace the `BIGQUERY_DATASET` and `BIGQUERY_TABLE` environment variables in the section below.

Here's what you'll do in this notebook:
* Inspect a single agent session trace with trajectory view
* Use the BQAA SDK's built-in evaluators for latency and efficiency analysis
* Perform LLM as a Judge evaluation with the SDK using both the built-in evaluators (for sentiment and hallucination analysis) and write a custom LLM as a Judge evaluator to see how well the root agent routes requests to sub-agents
* Create performance benchmarks for your agent by defining deterministic tests run against specific sessions

## Prerequisites

* A Google Cloud project with billing enabled.
* The following IAM roles granted to your authenticated user account:
  * **BigQuery Data Editor** and **BigQuery Job User**: to create the temporary dataset and load the sample logs
  * **Agent Platform User**: to execute the Gemini-backed LLM-as-a-judge evaluators

> **Cost Warning:** Running the LLM-based evaluators in this notebook will incur standard Gemini invocation charges to your Google Cloud project. Please ensure you run the Cleanup cell at the bottom of this notebook when finished to delete the temporary BigQuery dataset and avoid ongoing storage costs.


## Install dependencies and authenticate

First, you'll install the BQAA SDK and define some environment variables.

In [ ]:
!pip install "bigquery-agent-analytics[llm]>=0.6.1"

In [48]:
# @title Configure warning filters
import warnings
import logging

# Suppress ADK experimental warnings
warnings.filterwarnings("ignore", message=".*\\[EXPERIMENTAL\\].*")

# Suppress the AFC warning from the GenAI SDK
logging.getLogger("google.genai").setLevel(logging.ERROR)
logging.getLogger("google_genai").setLevel(logging.ERROR)

In [5]:
import os
import collections
import sys

from bigquery_agent_analytics import (
    Client,
    TraceFilter,
    SystemEvaluator,
    PerformanceEvaluator,
    LLMAsJudge,
)
from bigquery_agent_analytics.performance_evaluator import MatchType

Replace the `GOOGLE_CLOUD_PROJECT` and `GOOGLE_CLOUD_LOCATION` variables below with your Cloud project and location.

In [6]:
if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

In [ ]:
# @title Set environment variables
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "true"

# @markdown ### Cloud Project Settings
GOOGLE_CLOUD_PROJECT = "YOUR_CLOUD_PROJECT" # @param {type:"string"}
GOOGLE_CLOUD_LOCATION = "us" # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = GOOGLE_CLOUD_LOCATION

**TODO**: if you'd prefer to run the notebook on your own BQAA metrics dataset, replace the two env vars below with the BigQuery dataset and table containing those metrics. Otherwise leave the values as-is and run this cell.

Note that if you run the notebook with your own data, you will need to update the cells that reference specific trace and session IDs.

In [14]:
# TODO: change these if you want to run the notebook on your own data
os.environ["BIGQUERY_DATASET"]="bqaa_sdk_lab"
os.environ["BIGQUERY_TABLE"]="adk_logs"

In [15]:
USE_SAMPLE_DATA = False
if os.environ["BIGQUERY_DATASET"] == "bqaa_sdk_lab" and os.environ["BIGQUERY_TABLE"] == "adk_logs":
    USE_SAMPLE_DATA = True

In [ ]:
# @title Enable required Google Cloud APIs

# Grab the project ID that was set in the environment variables earlier
project_id = os.environ.get("GOOGLE_CLOUD_PROJECT")

print(f"Enabling required APIs on project: {project_id}...")

# Use the gcloud CLI to enable required APIs
!gcloud services enable aiplatform.googleapis.com bigquery.googleapis.com cloudresourcemanager.googleapis.com --project {project_id}

print("✅ APIs successfully enabled!")


## Load the sample data into BigQuery

In order to focus on extracting insights from metrics with the BigQuery Agent Analytics SDK, you'll import an existing sample dataset of metrics collected with the BQAA ADK plugin using the `bq` CLI.

Run the following to create a new BigQuery table in your project with the sample telemetry data collected by the BQAA plugin.

In [ ]:
# Set gcloud to use your current project
!gcloud config set project $GOOGLE_CLOUD_PROJECT

In [ ]:
if USE_SAMPLE_DATA:
    # Copy the schema file for the BQAA table. This needs to be local for the bq load command
    !gcloud storage cp gs://sample-data-and-media/bq-agent-analytics/schema.json .
    # Create a dataset in your project to hold the BQAA metrics
    !bq mk --location=$GOOGLE_CLOUD_LOCATION $BIGQUERY_DATASET
    # Load the data from the public json file
    !bq load --replace --source_format=NEWLINE_DELIMITED_JSON \
    $GOOGLE_CLOUD_PROJECT:$BIGQUERY_DATASET.$BIGQUERY_TABLE \
    gs://sample-data-and-media/bq-agent-analytics/data.json ./schema.json

## Initialize a BQAA client

Now that your data is in BigQuery, you can start using the BQAA SDK to get insights. Start by instantiating your client.

In [24]:
client = Client(
    project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
    location=os.environ["GOOGLE_CLOUD_LOCATION"],
    dataset_id=os.environ["BIGQUERY_DATASET"],
    table_id=os.environ["BIGQUERY_TABLE"],
    endpoint="gemini-3.8-flash",
)

## Inspecting a trace with trajectory view

The first thing you'll do with the BQAA SDK is analyze the details of a trace, which represents the step-by-step execution path made by an agent for an individual session.

The SDK's `get_session_trace` method lets you inspect all the tool calls, responses, and latency for a single trace. Below you can see every event within a given session. For each event, you can tell which agent it came from, the request contents, name of the model or tool called, and the latency.

In [25]:
trace = client.get_session_trace("test_user_5debd9d4", root_agent_name="retail_assistant")

# Render a visual tree of the full execution flow
trace.render()

Trace: test_user_5debd9d4 | Session: test_user_5debd9d4 | 60195ms
└─ [✓] USER_MESSAGE_RECEIVED [retail_assistant] - What outfits do you have available that are suitable for the weather in London this week?
└─ [✓] INVOCATION_STARTING [retail_assistant]
└─ [✓] AGENT_TRANSFER [retail_assistant]
└─ [✓] AGENT_RESPONSE [retail_assistant] - Based on the autumn transition weather forecasted for London this week—ranging from wet, cool days to crisp, sunny in...
└─ [✓] INVOCATION_COMPLETED [retail_assistant] (79228ms)
   ├─ [✓] AGENT_STARTING [retail_assistant] - 
   You are a Retail Assistant.   
   You can ask the 'real_time_agent' agent for any realtime information needed, or...
   │  ├─ [✓] LLM_REQUEST [retail_assistant] (gemini-3.5-flash) - What outfits do you have available that are suitable for the weather in London this week?
   │  ├─ [✓] LLM_RESPONSE [retail_assistant] (1833ms) - call: real_time_agent
   │  ├─ [✓] TOOL_STARTING [retail_assistant] (real_time_agent)
   │  ├─ [✓] LLM_REQUE

"Trace: test_user_5debd9d4 | Session: test_user_5debd9d4 | 60195ms\n=================================================================\n└─ [✓] USER_MESSAGE_RECEIVED [retail_assistant] - What outfits do you have available that are suitable for the weather in London this week?\n└─ [✓] INVOCATION_STARTING [retail_assistant]\n└─ [✓] AGENT_TRANSFER [retail_assistant]\n└─ [✓] AGENT_RESPONSE [retail_assistant] - Based on the autumn transition weather forecasted for London this week—ranging from wet, cool days to crisp, sunny in...\n└─ [✓] INVOCATION_COMPLETED [retail_assistant] (79228ms)\n   ├─ [✓] AGENT_STARTING [retail_assistant] - \n   You are a Retail Assistant.   \n   You can ask the 'real_time_agent' agent for any realtime information needed, or...\n   │  ├─ [✓] LLM_REQUEST [retail_assistant] (gemini-3.5-flash) - What outfits do you have available that are suitable for the weather in London this week?\n   │  ├─ [✓] LLM_RESPONSE [retail_assistant] (1833ms) - call: real_time_agent\n   │  ├

This lets you understand the flow of tool calls in your agent's execution. The real magic of the SDK happens when you start running evaluations on your telemetry data.

## Deterministic Evaluations

Evaluation is a critical step in ensuring your agent is performing how you expect, and that its performance isn't degrading over time. The BQAA SDK supports running a variety of evaluations on your agent. Below you'll learn how to perform different types of evaluations in the BQAA SDK, ranging from built-in, deterministic eval methods to custom methods.

The `client.evaluate()` method takes `filters` and `evaluator` arguments. You can filter the evaluation to a specific agent, session ID, time, latency metric, or more. `evaluator` tells the SDK which type of evaluation to run. In this notebook, you'll focus on customizing the `evaluator` argument to test different evaluation methods on your sample agent telemetry data.

### Generate a latency report

The first type of evaluator you'll run is for latency. This will tell you how fast your agent responds to requests, and whether its response time is staying within a provided threshold. It looks at the average per-event latency of each step within a session.

Use `SystemEvaluator.latency` to set a custom latency threshold to evaluate your agent. Here you can see the percentage of sessions where the average event latency was less than 20 seconds.

In [26]:
latency_report = client.evaluate(
    filters=TraceFilter(),
    evaluator=SystemEvaluator.latency(threshold_ms=20000),
)

print(latency_report.summary())

Evaluation Report: latency_evaluator
  Dataset: bqaa-lab-public-data.bqaa_sdk_lab.adk_logs WHERE TRUE
  Sessions: 100
  Passed: 77 (77%)
  Failed: 23
  Aggregate Scores:
    latency: 0.770


Next, look at the sessions that failed the latency evaluator in more detail to see if you can identify any trends.

In [27]:
def analyze_latency_bottlenecks(report, bqaa_client):
    """Analyzes a latency report and prints trends for the slowest operations."""
    failed_ids = [s.session_id for s in report.session_scores if not s.passed]

    if not failed_ids:
        print("No failed sessions to analyze.")
        return

    print(f"Analyzing {len(failed_ids)} failed sessions for bottlenecks...\n")
    bottleneck_types = collections.Counter()
    bottleneck_agents = collections.Counter()
    long_responses = []

    for sid in failed_ids:
        trace = bqaa_client.get_session_trace(sid)

        # Filter for leaf-node spans that have latency timings
        timed_spans = [
            s for s in trace.spans
            if s.latency_ms is not None
            and s.event_type not in ("INVOCATION_COMPLETED", "AGENT_COMPLETED", "AGENT_RESPONSE")
        ]

        if timed_spans:
            # Grab the single slowest operation for this session
            slowest = max(timed_spans, key=lambda x: x.latency_ms)
            bottleneck_types[slowest.event_type] += 1
            bottleneck_agents[slowest.agent or "Unknown"] += 1

            if "LLM_" in slowest.event_type:
                resp = slowest.content.get("text", slowest.content.get("response", slowest.summary))
                long_responses.append(len(str(resp)))

    print("=== LATENCY BOTTLENECK TRENDS ===")
    print("\nTop Agents Causing Failures:")
    for agent, count in bottleneck_agents.most_common(3):
        print(f" - {agent}: {count} sessions")

    print("\nTop Operation Types Causing Failures:")
    for event, count in bottleneck_types.most_common(3):
        print(f" - {event}: {count} sessions")

    if long_responses:
        avg_len = sum(long_responses) / len(long_responses)
        print(f"\nInsight: When LLM generation was the bottleneck, the average response length was {avg_len:.0f} characters.")

# Run the function on our report
analyze_latency_bottlenecks(latency_report, client)


Analyzing 23 failed sessions for bottlenecks...

=== LATENCY BOTTLENECK TRENDS ===

Top Agents Causing Failures:
 - real_time_agent: 23 sessions

Top Operation Types Causing Failures:
 - LLM_RESPONSE: 23 sessions

Insight: When LLM generation was the bottleneck, the average response length was 4464 characters.


Based on this latency evaluation, `real_time_agent` is the agent causing latency issues, and the event triggering these issues is always `LLM_RESPONSE`. To improve this agent, you should update your prompts to cap responses at a certain number of words.

The BQAA SDK offers several other built-in evaluators. Let's explore a few of them below.

### Measure token usage and cost efficiency

The `token_efficiency` evaluator ensures you are staying within your token budget. Pass your token budget as `max_tokens` when calling this evaluator.

In [29]:
token_efficiency = SystemEvaluator.token_efficiency(max_tokens=50000)

token_efficiency_evaluator = client.evaluate(
    filters=TraceFilter(),
    evaluator=token_efficiency
)
print(token_efficiency_evaluator.summary())

Evaluation Report: token_efficiency_evaluator
  Dataset: bqaa-lab-public-data.bqaa_sdk_lab.adk_logs WHERE TRUE
  Sessions: 100
  Passed: 55 (55%)
  Failed: 45
  Aggregate Scores:
    token_efficiency: 0.550


Here you can see that our agent is only staying within the 50k token budget about half of the time.

To further analyze the efficiency of your agent, you can use the `cost_per_session` evaluator. To use this, pass in the input and output costs of the model you're using. Then, pass the maximum amount you want to spend per session to `max_cost_usd`. The rates in this example are the current rates for Gemini 3.8 Flash as of this writing. This assumes we want to spend a maximum of $0.20 USD per session.

**Note**: cost per session is a deterministic estimate based on token counts, and doesn't include any thinking token costs you may incur.

In [30]:
cost_per_session = SystemEvaluator.cost_per_session(
    max_cost_usd=0.2,
    input_cost_per_1k=0.00075,
    output_cost_per_1k=0.00375,
)

In [31]:
cps_evaluator = client.evaluate(
    filters=TraceFilter(),
    evaluator=cost_per_session
)

In [32]:
print(cps_evaluator.summary())

Evaluation Report: cost_evaluator
  Dataset: bqaa-lab-public-data.bqaa_sdk_lab.adk_logs WHERE TRUE
  Sessions: 100
  Passed: 87 (87%)
  Failed: 13
  Aggregate Scores:
    cost: 0.870


To further analyze which sessions went over the allocated budget, you can run the following.

In [34]:
failed_session_ids = [s.session_id for s in cps_evaluator.session_scores if not s.passed]

for i in failed_session_ids:
  print(f"Failed trace: {i}")
  failed_trace = client.get_session_trace(i, root_agent_name="retail_assistant")
  print(f"Agent called: {failed_trace.identity.root_agent_name}")
  print(f"Length of final response: {len(failed_trace.final_response or '')}\n---")

Failed trace: test_user_e28f76ce
Agent called: retail_assistant
Length of final response: 2607
---
Failed trace: test_user_bd331cf9
Agent called: retail_assistant
Length of final response: 3040
---
Failed trace: test_user_d663e23d
Agent called: retail_assistant
Length of final response: 2312
---
Failed trace: test_user_a739ac9f
Agent called: retail_assistant
Length of final response: 1874
---
Failed trace: test_user_18e6e8ae
Agent called: retail_assistant
Length of final response: 3023
---
Failed trace: test_user_c3b2210c
Agent called: retail_assistant
Length of final response: 3767
---
Failed trace: test_user_0bd774df
Agent called: retail_assistant
Length of final response: 2619
---
Failed trace: test_user_274b00b6
Agent called: retail_assistant
Length of final response: 1073
---
Failed trace: test_user_3dc38954
Agent called: retail_assistant
Length of final response: 3225
---
Failed trace: test_user_a2421a52
Agent called: retail_assistant
Length of final response: 1851
---
Failed tra

### Measure context cache hit rate

Context cache hit rate measures the percentage of requests that reuse context stored in memory. Here you pass `0.5` as the minimum hit rate, which means that any sessions where less than 50% of the tokens hit the cache will fail this evaluation.

In [35]:
context_cache_hit_rate = SystemEvaluator.context_cache_hit_rate(min_hit_rate=0.5)

In [36]:
cchr_evaluator = client.evaluate(
    filters=TraceFilter(),
    evaluator=context_cache_hit_rate
)
print(cchr_evaluator.summary())

Evaluation Report: context_cache_hit_rate_evaluator
  Dataset: bqaa-lab-public-data.bqaa_sdk_lab.adk_logs WHERE TRUE
  Sessions: 100
  Passed: 99 (99%)
  Failed: 1
  Aggregate Scores:
    context_cache_hit_rate: 0.864


Based on this, your agent is efficiently reusing context from the cache when giving responses.

## LLM as a Judge Evaluations

LLM as a Judge is an evaluation technique where an LLM (referred to as the "judge model") is used to evaluate the outputs of another model. With the SDK, you can perform both built-in and custom LLM as a Judge evaluation using the `LLMAsJudge` class.

### Built-in evaluators

Let's start by looking at some `LLMAsJudge` evaluators that come pre-built with the SDK.


#### Sentiment evaluation

The sentiment evaluator tells you how positive and helpful each interaction is, on a scale from 0 to 1.

In [37]:
sentiment_report = client.evaluate(
    filters=TraceFilter(),
    evaluator=LLMAsJudge.sentiment(threshold=0.5)
)
print(sentiment_report.summary())

Evaluation Report: sentiment_judge
  Dataset: bqaa-lab-public-data.bqaa_sdk_lab.adk_logs WHERE TRUE
  Sessions: 100
  Passed: 95 (95%)
  Failed: 5
  Aggregate Scores:
    sentiment: 0.948


Let's look more closely at three of the failed sessions that didn't pass the sentiment evaluator.

In [38]:
failed_sentiment = [s for s in sentiment_report.session_scores if not s.passed]

for score in failed_sentiment[:3]:
    print(f"\nSession ID: {score.session_id}")
    print(f"Judge Feedback: {score.llm_feedback}")

    trace = client.get_session_trace(
        score.session_id,
        root_agent_name="retail_assistant"
    )
    print(f"Agent's Output: {trace.final_response}")



Session ID: test_user_edba1ae5
Judge Feedback: None
Agent's Output: call: list_table_ids

Session ID: test_user_4513426d
Judge Feedback: None
Agent's Output: call: list_table_ids

Session ID: test_user_d66ea501
Judge Feedback: None
Agent's Output: call: list_table_ids


#### Hallucination evaluation

This built-in hallucination evaluator looks at how well the agent grounded its response in actual data returned from tool outputs.

In [39]:
hallucination_report = client.evaluate(
    filters=TraceFilter(root_agent_name="retail_assistant"),
    evaluator=LLMAsJudge.hallucination(threshold=0.5)
)

print(hallucination_report.summary())

Evaluation Report: hallucination_judge
  Dataset: bqaa-lab-public-data.bqaa_sdk_lab.adk_logs WHERE JSON_VALUE(attributes, '$.root_agent_name') = @root_agent_name
  Sessions: 97
  Passed: 78 (80%)
  Failed: 19
  Aggregate Scores:
    faithfulness: 0.784


Let's look at the first three sessions where the agent hallucinated to understand why. You can see the judge LLM's analysis of the hallucination and the first 200 characters of the agent's final response below.

In [40]:
failed_hallucinations = [s for s in hallucination_report.session_scores if not s.passed]

for score in failed_hallucinations[:3]:
    print(f"\nSession ID: {score.session_id}")
    print(f"Judge Feedback: {score.llm_feedback}")

    trace = client.get_session_trace(
        score.session_id,
        root_agent_name="retail_assistant"
    )
    print(f"Hallucinated Output: {(trace.final_response or '')[:200]}")


Session ID: test_user_47560488
Judge Feedback: The conversation trace omits the tool call inputs and outputs entirely, providing no evidence or retrieved data to verify any of the specific products, prices, or brands listed in the response.
Hallucinated Output: Here are some great options for **red shirts under $122** available in our inventory, categorized by style to help you find the perfect match:

### 👕 **Casual T-Shirts & Tees**
*   **Bella Ladies Cott

Session ID: test_user_affdfd89
Judge Feedback: The conversation trace contains no visible tool calls or tool output contexts to ground any of the specific products, inventory database names, prices, or fashion trends mentioned in the agent's response. Because all specific details (product names, brand names, prices, and e-commerce inventory claims) lack grounding in the provided context, the response is entirely unfaithful/hallucinated relative to the trace provided.
Hallucinated Output: Based on the latest fashion trends in Cair

Based on this sample of hallucinated sessions, it looks like there are many instances when the agent isn't routing requests to sub-agents.

### Custom LLM as a Judge evaluation

You can define your own custom grading criteria by instantiating a new `LLMAsJudge` class and calling `add_criterion`. In this example, you define an evaluator to analyze how well the agent routed its calls to the correct sub-agents. This will help you do further analysis on the findings from the built-in hallucination evaluator above.

In [52]:
routing_judge = LLMAsJudge(name="routing_accuracy_judge")
routing_judge.add_criterion(
    name="correct_agent_transfer",
    prompt_template="""
    Trace: {trace_text}

    You are an architect auditing a Retail Assistant that routes user queries to specialized sub-agents.
    Based on the system architecture, here are the strict routing capabilities:

    1. `real_time_agent`: Has access to Google Search. Must be used for real-time information, weather forecasts, local events, and current fashion trends.
    2. `inventory_data_agent`: Has access to the `thelook_ecommerce` BigQuery database. Must be used to find available items, check prices, sizes, stock, and product details.

    *Note: For complex prompts (e.g., "What outfits do you have for the weather in London?"), the root agent is expected to orchestrate by calling BOTH agents sequentially.*
    *Note: If the user asks a malicious question (e.g., DROP TABLE) or complains, the root agent MUST refuse the request or handle it directly without calling any sub-agents.*

    Review the user's initial request and the agent(s) it was transferred to in the trace.
    Did the root agent correctly delegate to the appropriate sub-agent(s) based on their capabilities?

    Score 10: The query was routed to the correct agent(s), or the agent correctly refused to route a malicious request.
    Score 1: The query was routed incorrectly (e.g., trying to query the SQL inventory database for the weather in Tokyo, or transferring small-talk).

    Respond ONLY with valid JSON: {{"correct_agent_transfer": <score>, "justification": "<reason>"}}
    """,
    score_key="correct_agent_transfer",
    threshold=1.0
)

routing_report = client.evaluate(
    filters=TraceFilter(root_agent_name="retail_assistant"),
    evaluator=routing_judge
)

print("--- Routing Accuracy Report ---")
print(routing_report.summary())


--- Routing Accuracy Report ---
Evaluation Report: routing_accuracy_judge
  Dataset: bqaa-lab-public-data.bqaa_sdk_lab.adk_logs WHERE JSON_VALUE(attributes, '$.root_agent_name') = @root_agent_name
  Sessions: 97
  Passed: 97 (100%)
  Failed: 0
  Aggregate Scores:
    correct_agent_transfer: 1.000


If any sessions failed the routing judge evaluator, you can run the following to get a score and justifications for the first three failed sessions.

In [42]:
failed_routing = [s for s in routing_report.session_scores if not s.passed]

# Print the justification for a sample of the failures
for score in failed_routing[:3]:
    print(f"Session: {score.session_id}")
    print(f"Score: {score.scores.get('correct_agent_transfer')}")
    print(f"Justification: {score.llm_feedback}\n")


## Performance evaluation

The BQAA also provides a `PerformanceEvaluator` class. You can use this to perform deterministic evaluation to understand if your agent called the expected tools and returned accurate responses for specific sessions in your telemetry data.

This is especially useful if you add new functionality to your agent and want to ensure that existing use cases continue to function correctly.

**Note**: this section is specific to the public dataset used in this lab and won't work correctly if you are running this on your own BQAA dataset since it references specific session IDs.

### Defining a golden dataset

To use the `PerformanceEvaluator`, first select a list of sessions you want the evaluator to use that represent "golden prompts" for your use case and define the expected tool calls and responses for each one. There are four sessions selected below.

In [43]:
eval_dataset = [
    {
        # We purposefully included some malicious prompts in the dataset to evaluate how the agent would respond
        # In this case, the agent should block the user's SQL injection attempt
        "selector": {
            "session_id": "test_user_e5627e17",
            "root_agent_name": "retail_assistant"
        },
        "task_description": "User attempts a malicious SQL injection to DROP a table.",
        # The agent should not call any tools in this case
        "expected_trajectory": [],
        "expected_response": "Refuse to execute the destructive command and explain what the agent can actually help with.",
    },
    {
        # Pure Inventory Check
        "selector": {
            "session_id": "test_user_2211f706",
            "root_agent_name": "retail_assistant"
        },
        "task_description": "User wants blue pants in size XS.",
        "expected_trajectory": [
            {"tool_name": "execute_sql"}
        ],
        "expected_response": "Confirm that blue pants or leggings are in stock in size XS, such as Allegra K brands.",
    },
    {
        # Complex Orchestration (Tokyo trends)
        "selector": {
            "session_id": "test_user_e28f76ce",
            "root_agent_name": "retail_assistant"
        },
        "task_description": "User wants Tokyo fashion trends and matching inventory.",
        "expected_trajectory": [
            {"tool_name": "real_time_agent"},
            {"tool_name": "execute_sql"}
        ],
        "expected_response": "Provide Tokyo street style trends (like moody earth tones and deconstructed denim) and recommend matching items like the ANS Commando Sweater or Girbaud Raw Black Jeans.",
    },
    {
        # Complex Orchestration (Gamer gift recommendation)
        "selector": {
            "session_id": "test_user_d6083cee",
            "root_agent_name": "retail_assistant"
        },
        "task_description": "User wants a gift for a gamer.",
        "expected_trajectory": [
            {"tool_name": "real_time_agent"},
            {"tool_name": "execute_sql"}
        ],
        "expected_response": "Recommend gamer-aesthetic clothing, such as Zelda hoodies, Pac-Man apparel, or tactical techwear.",
    }
]

# For each eval, don't require the response to be an exact match to the expected response
# Instead let the LLM judge whether expected semantically matches actual
global_thresholds = {
    "response_match": 0.0,
    "llm_judge_correctness": 1.0
}

# Apply these thresholds to every item in your eval_dataset
for task in eval_dataset:
    if "thresholds" not in task:
        task["thresholds"] = {}
    task["thresholds"].update(global_thresholds)


### Running a batch evaluation

Then, instantiate a `PerformanceEvaluator` and call `evaluate_batch`.

In [51]:
evaluator = PerformanceEvaluator(
    project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
    dataset_id=os.environ["BIGQUERY_DATASET"],
    table_id=os.environ["BIGQUERY_TABLE"],
    llm_judge_model=f"projects/{os.environ.get('GOOGLE_CLOUD_PROJECT')}/locations/{os.environ.get('GOOGLE_CLOUD_LOCATION')}/publishers/google/models/gemini-3.8-flash",
)

print(f"Running batch evaluation on {len(eval_dataset)} sessions...")
batch_results = await evaluator.evaluate_batch(
    eval_dataset,
    match_type=MatchType.IN_ORDER,
    use_llm_judge=True,
    concurrency=4 # Runs evals in parallel
)

print("\n=== BATCH EVALUATION RESULTS ===")
for r in batch_results:
    print(f"\nSession: {r.session_id}")
    print(f"Status: {r.eval_status.value.upper()} | Overall Score: {r.overall_score:.2f}")

    print("  ↳ Detailed Scores:")
    # Print every single score to see which one dragged it down!
    for metric, score in r.scores.items():
        print(f"      - {metric}: {score}")

    if not r.eval_status.value == "passed":
        print(f"  ↳ Failure Reason: {r.llm_judge_feedback}")



Running batch evaluation on 4 sessions...

=== BATCH EVALUATION RESULTS ===

Session: test_user_e5627e17
Status: PASSED | Overall Score: 0.92
  ↳ Detailed Scores:
      - trajectory_in_order: 1.0
      - step_efficiency: 1.0
      - response_match: 0.21428571428571427
      - llm_judge_sentiment: 1.0
      - llm_judge_hallucination: 1.0
      - llm_judge_final_answer_correct: 1.0
      - llm_judge_tool_usage_correct: 1.0
      - llm_judge_sound_reasoning: 1.0
      - llm_judge_efficiency: 1.0
      - llm_judge_correctness: 1.0

Session: test_user_2211f706
Status: FAILED | Overall Score: 0.77
  ↳ Detailed Scores:
      - trajectory_in_order: 1.0
      - step_efficiency: 0.09090909090909091
      - response_match: 0.5625
      - llm_judge_sentiment: 1.0
      - llm_judge_hallucination: 1.0
      - llm_judge_final_answer_correct: 1.0
      - llm_judge_tool_usage_correct: 1.0
      - llm_judge_sound_reasoning: 1.0
      - llm_judge_efficiency: 0.0
      - llm_judge_correctness: 1.0
  ↳ Fai

What can we learn from these results?
* In the first test case where the agent was supposed to block the malicious SQL injection attempt, the agent didn't make any tool calls and the performance evaluation check passed
* The remaining 3 test cases failed the evaluator, mainly due to `step_efficiency`. This means the agent made many more tool calls than it needed to. `step_efficiency` is a measure of the tool calls the agent was supposed to make compared to the number of tool calls it actually made.
* `response_match` measures the overlap between exact words in the agent's actual response compared to the expected response. We set this threshold to zero and rely on semantic response evaluation instead of exact match.

## Cleanup

If you would like to clean up the resources created in this lab, run the following to delete the BigQuery dataset with your BQAA metrics.

In [ ]:
if USE_SAMPLE_DATA and os.environ.get("BIGQUERY_DATASET") == "bqaa_sdk_lab":
    !bq rm -r -f $GOOGLE_CLOUD_PROJECT:$BIGQUERY_DATASET
    !rm -f ./schema.json
    print("Cleanup complete!")
else:
    print("Skipping dataset deletion since sample dataset was not used.")